# Phase 2 — Classical ML
## Day 08: Ensembles & XGBoost
**Date:** April 24, 2026

### Learning Objectives
- Understand bagging vs boosting (and why they work differently)
- Build a Random Forest and tune key hyperparameters
- Use XGBoost for tabular classification
- Combine models with VotingClassifier and StackingClassifier


In [ ]:
# Setup — run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    VotingClassifier,
    StackingClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")

print("All imports OK")


In [ ]:
# Create synthetic dataset — binary classification, 1000 samples
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class balance: {np.bincount(y_train)}")


## 1. Bagging — Bootstrap Aggregating

**Bagging** trains many models *in parallel*, each on a random subset of the training data (sampled with replacement — that's "bootstrap"). Then it averages (or votes) their predictions.

The key insight: if each model makes different errors, averaging them out reduces variance. This is why bagging works best when your base model is high-variance (like a deep decision tree).

**Random Forest** is bagging + an extra trick: at each split, it only considers a *random subset of features*. This makes the trees even more different from each other, which helps even more.


In [ ]:
# BaggingClassifier — wrap any base estimator
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=None),  # unpruned trees (high variance)
    n_estimators=50,
    max_samples=0.8,   # use 80% of data per tree
    max_features=0.8,  # use 80% of features per tree
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train, y_train)
print(f"Bagging accuracy: {accuracy_score(y_test, bagging.predict(X_test)):.4f}")


In [ ]:
# Random Forest — scikit-learn's optimized bagging + feature randomness
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,       # let trees grow deep (bagging controls variance)
    max_features='sqrt',  # at each split, consider sqrt(n_features) features
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print(f"Random Forest accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}")


In [ ]:
# Effect of n_estimators — more trees = more stable but slower
scores = []
for n in [1, 5, 10, 25, 50, 100, 200]:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    score = cross_val_score(rf_n, X_train, y_train, cv=3).mean()
    scores.append((n, score))
    print(f"  n_estimators={n:4d} -> CV accuracy={score:.4f}")


## 2. Boosting — Sequential Error Correction

**Boosting** trains models *sequentially*. Each new model focuses on the examples the previous models got wrong. The models are weak (usually shallow trees), but together they form a strong learner.

Key differences from bagging:
- Bagging: parallel, reduces variance. Boosting: sequential, reduces bias.
- Bagging needs deep trees. Boosting needs *shallow* trees (low depth = weak learner).
- Boosting can overfit if you use too many estimators or too high a learning rate.

**XGBoost** is an optimized gradient boosting library — faster, regularized, and usually gives the best results on structured/tabular data.


In [ ]:
# sklearn's GradientBoostingClassifier — built-in, no extra install
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,   # shrinkage — smaller = slower but more robust
    max_depth=3,          # shallow trees are key for boosting
    subsample=0.8,        # stochastic boosting (sample 80% of data per round)
    random_state=42
)
gb.fit(X_train, y_train)
print(f"GradientBoosting accuracy: {accuracy_score(y_test, gb.predict(X_test)):.4f}")


In [ ]:
# XGBoost — usually the go-to for tabular data competitions
try:
    import xgboost as xgb
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,  # fraction of features per tree (like max_features in RF)
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    print(f"XGBoost accuracy: {accuracy_score(y_test, xgb_model.predict(X_test)):.4f}")
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    xgb_model = None


In [ ]:
# XGBoost feature importance
try:
    importances = xgb_model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:10]
    print("Top 10 features by XGBoost importance:")
    for i, idx in enumerate(top_idx):
        print(f"  Feature {idx:2d}: {importances[idx]:.4f}")
except:
    print("Skipped (XGBoost not available)")


## 3. Learning Rate vs n_estimators Tradeoff

In boosting, `learning_rate` and `n_estimators` work together:
- Low learning rate = each tree contributes less = need more trees to converge
- High learning rate = risk of overfitting quickly

A good rule of thumb: start with `learning_rate=0.1, n_estimators=100`, then lower the rate and raise n_estimators if you want to squeeze more performance.


In [ ]:
# Show the tradeoff: lower lr needs more estimators
configs = [
    (0.3, 50),
    (0.1, 100),
    (0.05, 200),
    (0.01, 500),
]
print(f"{'LR':>6} {'n_est':>6} {'CV Accuracy':>12}")
print("-" * 28)
for lr, n_est in configs:
    model = GradientBoostingClassifier(
        learning_rate=lr, n_estimators=n_est, max_depth=3,
        subsample=0.8, random_state=42
    )
    score = cross_val_score(model, X_train, y_train, cv=3).mean()
    print(f"{lr:>6.2f} {n_est:>6d} {score:>12.4f}")


## 4. VotingClassifier — Combining Different Models

When you have several good models that make *different* kinds of errors, you can combine them with a VotingClassifier.

- **Hard voting**: each model votes for a class, majority wins.
- **Soft voting**: average the predicted probabilities, pick the highest. Usually better when models are calibrated.

The trick is to combine models that are *diverse* — a tree-based model + a linear model tend to complement each other well.


In [ ]:
# Soft voting: average probabilities from 3 different model types
lr_clf = LogisticRegression(max_iter=500, random_state=42)
rf_clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
gb_clf = GradientBoostingClassifier(n_estimators=50, random_state=42)

# Individual scores first
for name, clf in [("LogReg", lr_clf), ("RandomForest", rf_clf), ("GradBoost", gb_clf)]:
    score = cross_val_score(clf, X_train, y_train, cv=3).mean()
    print(f"  {name}: {score:.4f}")

voter_soft = VotingClassifier(
    estimators=[("lr", lr_clf), ("rf", rf_clf), ("gb", gb_clf)],
    voting='soft',  # use predicted probabilities
    n_jobs=-1
)
score_soft = cross_val_score(voter_soft, X_train, y_train, cv=3).mean()
print(f"  Soft Voting ensemble: {score_soft:.4f}")


In [ ]:
# Hard voting comparison
voter_hard = VotingClassifier(
    estimators=[("lr", lr_clf), ("rf", rf_clf), ("gb", gb_clf)],
    voting='hard',
    n_jobs=-1
)
score_hard = cross_val_score(voter_hard, X_train, y_train, cv=3).mean()
print(f"Hard Voting: {score_hard:.4f}")
print(f"Soft Voting: {score_soft:.4f}")
print("Soft voting usually wins when models output probabilities.")


## 5. StackingClassifier — Meta-Learning

**Stacking** takes it one step further. Instead of just averaging, it trains a *meta-model* (also called a blender) that *learns* how to best combine the base models.

How it works:
1. Split training data with cross-validation.
2. Train base models on each fold, predict on the held-out fold.
3. Use those out-of-fold predictions as features for the meta-model.
4. The meta-model learns which base model to trust for which inputs.

This is more powerful than voting but also slower to train and easier to overfit.


In [ ]:
# StackingClassifier — base learners feed into a LogisticRegression meta-model
stacking = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
        ("gb", GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ],
    final_estimator=LogisticRegression(),  # meta-model
    cv=3,                                   # cross-val folds for generating meta-features
    passthrough=False,                      # don't pass original features to meta-model
    n_jobs=-1
)
score_stack = cross_val_score(stacking, X_train, y_train, cv=3).mean()
print(f"Stacking CV accuracy: {score_stack:.4f}")


## Tricky Bits — Common Mistakes


In [ ]:
# Mistake 1: Using a high learning_rate with many estimators in boosting -> overfit
gb_overfit = GradientBoostingClassifier(
    learning_rate=1.0,   # way too high
    n_estimators=500,    # way too many
    max_depth=5,
    random_state=42
)
gb_overfit.fit(X_train, y_train)
train_acc = accuracy_score(y_train, gb_overfit.predict(X_train))
test_acc = accuracy_score(y_test, gb_overfit.predict(X_test))
print(f"Train accuracy: {train_acc:.4f}  (memorizing training data)")
print(f"Test accuracy:  {test_acc:.4f}  (generalizes poorly)")
print("Solution: lower learning_rate, add subsample, reduce n_estimators")


In [ ]:
# Mistake 2: Voting with models that aren't calibrated — hard voting on bad probabilities
# Logistic Regression outputs well-calibrated probabilities.
# Decision trees do NOT — their probability estimates are overconfident.
# Soft voting with poorly calibrated models can hurt you.

dt_uncal = DecisionTreeClassifier(max_depth=2, random_state=42)  # shallow = ok probabilities
voter_mixed = VotingClassifier(
    estimators=[("dt", dt_uncal), ("lr", LogisticRegression(max_iter=200))],
    voting='soft'
)
score = cross_val_score(voter_mixed, X_train, y_train, cv=3).mean()
print(f"Soft voting (mixed calibration): {score:.4f}")
print("Always check if your models are calibrated before using soft voting.")


In [ ]:
# Mistake 3: Stacking without cross-val predictions -> data leakage
# Wrong way: train base models on all data, then use their predictions as features
# for the meta-model trained on the same data. The meta-model sees "cheated" predictions.
# Right way: sklearn's StackingClassifier handles this automatically using cv= parameter.
print("StackingClassifier with cv=3 uses out-of-fold predictions — no leakage.")
print("If you build stacking manually, always use cross_val_predict() for meta-features.")


## Trick Questions

<details>
<summary>Q1: Random Forest uses bagging. Does that mean more trees always means better accuracy?</summary>

**Not exactly.** More trees reduce variance, so accuracy improves quickly at first. But after ~100-200 trees, the improvement plateaus. Adding more trees just makes training slower. There's no overfitting with more trees (unlike boosting), but there's diminishing returns.
</details>

<details>
<summary>Q2: In boosting, should you use deep or shallow trees?</summary>

**Shallow (max_depth=3-6).** Boosting corrects errors sequentially, so each tree only needs to be a weak learner. Deep trees in boosting lead to overfitting fast. This is the opposite of bagging, where you want deep trees.
</details>

<details>
<summary>Q3: Soft voting requires all estimators to have predict_proba(). What happens if one doesn't?</summary>

**sklearn raises an error.** For example, vanilla SVM without `probability=True` can't do soft voting. You'd either switch to hard voting or enable probability estimates in the model.
</details>

<details>
<summary>Q4: In stacking, why use cross-validation to generate meta-features instead of just predicting on the training set?</summary>

**To avoid data leakage.** If the base model is trained on the whole training set and then predicts on that same set, its predictions are overfitted and "too good." The meta-model then learns to trust predictions that won't be that good at test time. Cross-val out-of-fold predictions simulate what the models would predict on unseen data.
</details>

<details>
<summary>Q5: Bagging reduces variance. Boosting reduces bias. Which one would you use if your model is already overfitting?</summary>

**Bagging (or Random Forest).** If you're overfitting, you have high variance. Bagging is designed to reduce variance by averaging diverse models. Boosting would likely make overfitting worse because it keeps adding model capacity.
</details>


## Exercises — Fill in the Blanks

Each cell has `___` placeholders. Replace them and check the `assert` passes.


In [ ]:
# Exercise 1: Create a RandomForestClassifier with 200 trees, max_depth=10, random_state=0
# Then fit it and check test accuracy is above 0.85

rf_ex1 = RandomForestClassifier(n_estimators=___, max_depth=___, random_state=___)
rf_ex1.fit(X_train, y_train)
acc_ex1 = accuracy_score(y_test, rf_ex1.predict(X_test))
print(f"Exercise 1 accuracy: {acc_ex1:.4f}")
assert acc_ex1 > 0.85, f"Expected >0.85, got {acc_ex1:.4f}"
print("Exercise 1 passed!")


In [ ]:
# Exercise 2: Create a GradientBoostingClassifier with learning_rate=0.05,
# n_estimators=200, max_depth=3, subsample=0.8, random_state=42

gb_ex2 = GradientBoostingClassifier(
    learning_rate=___,
    n_estimators=___,
    max_depth=___,
    subsample=___,
    random_state=42
)
gb_ex2.fit(X_train, y_train)
acc_ex2 = accuracy_score(y_test, gb_ex2.predict(X_test))
print(f"Exercise 2 accuracy: {acc_ex2:.4f}")
assert acc_ex2 > 0.85, f"Expected >0.85, got {acc_ex2:.4f}"
print("Exercise 2 passed!")


In [ ]:
# Exercise 3: Get cross-validation score (cv=5) for the Random Forest from Exercise 1
# Store the mean score in cv_mean_ex3

cv_scores_ex3 = cross_val_score(rf_ex1, X_train, y_train, cv=___)
cv_mean_ex3 = cv_scores_ex3.___()
print(f"Exercise 3 CV mean: {cv_mean_ex3:.4f}")
assert cv_mean_ex3 > 0.80, f"Expected >0.80, got {cv_mean_ex3:.4f}"
print("Exercise 3 passed!")


In [ ]:
# Exercise 4: Build a soft VotingClassifier using:
# - RandomForestClassifier(n_estimators=50, random_state=42) named "rf"
# - GradientBoostingClassifier(n_estimators=50, random_state=42) named "gb"
# Use voting='soft'

voter_ex4 = VotingClassifier(
    estimators=[("rf", ___), ("gb", ___)],
    voting=___
)
voter_ex4.fit(X_train, y_train)
acc_ex4 = accuracy_score(y_test, voter_ex4.predict(X_test))
print(f"Exercise 4 accuracy: {acc_ex4:.4f}")
assert acc_ex4 > 0.85, f"Expected >0.85, got {acc_ex4:.4f}"
print("Exercise 4 passed!")


In [ ]:
# Exercise 5: Get feature importances from rf_ex1 and find the index of the most important feature
importances_ex5 = rf_ex1.___  # attribute name for feature importances
most_important_idx = np.argmax(importances_ex5)
print(f"Most important feature index: {most_important_idx}")
print(f"Importance value: {importances_ex5[most_important_idx]:.4f}")
assert 0 <= most_important_idx < X_train.shape[1], "Invalid feature index"
print("Exercise 5 passed!")


In [ ]:
# Exercise 6: Create a StackingClassifier with:
# - base estimators: rf (RandomForest 50 trees) and gb (GradBoost 50 trees)
# - meta-model: LogisticRegression()
# - cv=3
# Then cross-validate with cv=3 and check mean > 0.85

stacking_ex6 = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
        ("gb", GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ],
    final_estimator=___,
    cv=___
)
score_ex6 = cross_val_score(stacking_ex6, X_train, y_train, cv=3).mean()
print(f"Exercise 6 stacking CV: {score_ex6:.4f}")
assert score_ex6 > 0.85, f"Expected >0.85, got {score_ex6:.4f}"
print("Exercise 6 passed!")


In [ ]:
# Exercise 7: XGBoost — create XGBClassifier with n_estimators=100, max_depth=3,
# learning_rate=0.1, subsample=0.8, random_state=42
# Fit on training data, check test accuracy > 0.88

try:
    import xgboost as xgb
    xgb_ex7 = xgb.XGBClassifier(
        n_estimators=___,
        max_depth=___,
        learning_rate=___,
        subsample=___,
        eval_metric='logloss',
        random_state=42
    )
    xgb_ex7.fit(X_train, y_train)
    acc_ex7 = accuracy_score(y_test, xgb_ex7.predict(X_test))
    print(f"Exercise 7 XGBoost accuracy: {acc_ex7:.4f}")
    assert acc_ex7 > 0.88, f"Expected >0.88, got {acc_ex7:.4f}"
    print("Exercise 7 passed!")
except ImportError:
    print("XGBoost not installed — skipping exercise 7")


## Solutions

<details>
<summary>Exercise 1</summary>

```python
rf_ex1 = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=0)
```
</details>

<details>
<summary>Exercise 2</summary>

```python
gb_ex2 = GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=200,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
```
</details>

<details>
<summary>Exercise 3</summary>

```python
cv_scores_ex3 = cross_val_score(rf_ex1, X_train, y_train, cv=5)
cv_mean_ex3 = cv_scores_ex3.mean()
```
</details>

<details>
<summary>Exercise 4</summary>

```python
voter_ex4 = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
        ("gb", GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ],
    voting='soft'
)
```
</details>

<details>
<summary>Exercise 5</summary>

```python
importances_ex5 = rf_ex1.feature_importances_
most_important_idx = np.argmax(importances_ex5)
```
</details>

<details>
<summary>Exercise 6</summary>

```python
stacking_ex6 = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
        ("gb", GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ],
    final_estimator=LogisticRegression(),
    cv=3
)
```
</details>

<details>
<summary>Exercise 7</summary>

```python
xgb_ex7 = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    eval_metric='logloss',
    random_state=42
)
```
</details>


## Cumulative Review — Days 1-7

These exercises mix topics from all previous days.


In [ ]:
# Review 1 (Day 6): Create a sklearn Pipeline with StandardScaler + LogisticRegression
# Fit on X_train, y_train and check accuracy on X_test > 0.80
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe_rev1 = Pipeline([
    ("scaler", ___),
    ("clf", ___(max_iter=500, random_state=42)),
])
pipe_rev1.fit(X_train, y_train)
acc_rev1 = accuracy_score(y_test, pipe_rev1.predict(X_test))
print(f"Review 1 pipeline accuracy: {acc_rev1:.4f}")
assert acc_rev1 > 0.80
print("Review 1 passed!")


In [ ]:
# Review 2 (Day 6): Use StratifiedKFold(n_splits=5) and cross_val_score on rf_ex1
from sklearn.model_selection import StratifiedKFold

skf_rev2 = StratifiedKFold(n_splits=___)
scores_rev2 = cross_val_score(rf_ex1, X_train, y_train, cv=___)
print(f"Review 2 StratifiedKFold scores: {scores_rev2.round(3)}")
print(f"Mean: {scores_rev2.mean():.4f}")
assert len(scores_rev2) == 5
print("Review 2 passed!")


In [ ]:
# Review 3 (Day 7): Train a DecisionTreeClassifier with max_depth=5
# and print how many leaf nodes it has
from sklearn.tree import DecisionTreeClassifier

dt_rev3 = DecisionTreeClassifier(max_depth=___, random_state=42)
dt_rev3.fit(X_train, y_train)
n_leaves = dt_rev3.tree_.n_leaves
print(f"Review 3 leaf nodes: {n_leaves}")
assert n_leaves > 1, "Tree should have more than 1 leaf"
print("Review 3 passed!")


In [ ]:
# Review 4 (Day 2): Use numpy to compute the mean and std of each column in X_train
means = np.___( ___, axis=0)  # mean across rows (axis=0)
stds  = np.___( ___, axis=0)
print(f"Review 4 - First 5 means: {means[:5].round(3)}")
print(f"Review 4 - First 5 stds:  {stds[:5].round(3)}")
assert means.shape == (20,)
print("Review 4 passed!")


In [ ]:
# Review 5 (Day 1): Put X_train and y_train into a DataFrame.
# Add a column 'label' for y_train and show value_counts().

df_rev5 = pd.DataFrame(___, columns=[f"f{i}" for i in range(20)])
df_rev5["label"] = ___
print("Review 5 label counts:")
print(df_rev5["label"].value_counts())
assert "label" in df_rev5.columns
print("Review 5 passed!")


In [ ]:
# Review 6 (Day 3): Introduce 10% missing values into X_train randomly,
# then fill them with the column median using pandas.

df_rev6 = pd.DataFrame(X_train.copy(), columns=[f"f{i}" for i in range(20)])

# Introduce NaNs
mask = np.random.RandomState(0).rand(*df_rev6.shape) < 0.10
df_rev6[mask] = np.nan

print(f"Missing before: {df_rev6.isna().sum().sum()}")

# Fill with median
df_rev6_filled = df_rev6.fillna(df_rev6.___)
print(f"Missing after:  {df_rev6_filled.isna().sum().sum()}")
assert df_rev6_filled.isna().sum().sum() == 0
print("Review 6 passed!")


In [ ]:
# Review 7 (Day 7): Compare Decision Tree gini vs entropy on same data
for criterion in ["gini", "entropy"]:
    dt = DecisionTreeClassifier(criterion=___, max_depth=5, random_state=42)
    score = cross_val_score(dt, X_train, y_train, cv=3).mean()
    print(f"  {criterion}: {score:.4f}")
print("Review 7 passed — no assert, just observe the difference!")


## Cumulative Review Solutions

<details>
<summary>Review 1</summary>

```python
pipe_rev1 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=42)),
])
```
</details>

<details>
<summary>Review 2</summary>

```python
skf_rev2 = StratifiedKFold(n_splits=5)
scores_rev2 = cross_val_score(rf_ex1, X_train, y_train, cv=skf_rev2)
```
</details>

<details>
<summary>Review 3</summary>

```python
dt_rev3 = DecisionTreeClassifier(max_depth=5, random_state=42)
```
</details>

<details>
<summary>Review 4</summary>

```python
means = np.mean(X_train, axis=0)
stds  = np.std(X_train, axis=0)
```
</details>

<details>
<summary>Review 5</summary>

```python
df_rev5 = pd.DataFrame(X_train, columns=[f"f{i}" for i in range(20)])
df_rev5["label"] = y_train
```
</details>

<details>
<summary>Review 6</summary>

```python
df_rev6_filled = df_rev6.fillna(df_rev6.median())
```
</details>

<details>
<summary>Review 7</summary>

```python
dt = DecisionTreeClassifier(criterion=criterion, max_depth=5, random_state=42)
```
</details>


In [ ]:
# Cheat Sheet — Day 8 Quick Reference
cheat = """
=== Day 08: Ensembles & XGBoost — Cheat Sheet ===

BAGGING (RandomForest)
  RandomForestClassifier(
      n_estimators=100,    # more = more stable, diminishing returns after ~200
      max_features='sqrt', # features considered per split
      max_depth=None,      # let trees grow (bagging controls variance)
  )
  .feature_importances_   # array of importance per feature

GRADIENT BOOSTING (sklearn)
  GradientBoostingClassifier(
      n_estimators=100,    # more trees = lower bias, higher overfit risk
      learning_rate=0.1,   # shrinkage — lower = need more trees
      max_depth=3,         # keep trees SHALLOW for boosting
      subsample=0.8,       # stochastic boosting
  )

XGBOOST
  xgb.XGBClassifier(
      n_estimators=100, learning_rate=0.1, max_depth=3,
      subsample=0.8, colsample_bytree=0.8,
      eval_metric='logloss',
  )

VOTING
  VotingClassifier(estimators=[...], voting='soft')  # soft = use proba
  VotingClassifier(estimators=[...], voting='hard')  # hard = majority vote

STACKING
  StackingClassifier(
      estimators=[...],          # base models
      final_estimator=LogReg(),  # meta-model
      cv=5,                      # folds for out-of-fold predictions
  )

RULES OF THUMB
  Bagging:   use deep trees, reduces VARIANCE
  Boosting:  use shallow trees, reduces BIAS
  Both suffer from leakage if meta-features aren't cross-validated
  lr=0.1 + n_est=100 is a safe starting point for boosting
"""
print(cheat)


---
## Next Up: Day 9 — ClassificationMetrics

Tomorrow we go deep on evaluation:
- Confusion matrix and when accuracy is misleading
- Precision, recall, and F1 score
- ROC curves and AUC
- Gini coefficient
- Threshold tuning for imbalanced datasets
